# CYTools interface

> **What's in this notebook?** This notebook demonstrates how to interface `CYTools` within JAXVacua.
>
> (*Created:* Andreas Schachner, June 25, 2024)

## Outline

1. [Setup](#setup)
2. [CYTools introduction](#cytools-introduction)
3. [Grabbing models at LCS from the Kreuzer-Skarke list](#grabbing-models-at-lcs-from-the-kreuzer-skarke-list)
4. [Take-aways](#take-aways)
5. [Further reading](#further-reading)

## Setup

In [ ]:
# General imports
import warnings
import numpy as np
from tqdm.auto import tqdm

# JAX imports
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

# Plotting
import seaborn as sn
import matplotlib.pyplot as plt
cmap = sn.color_palette("viridis", as_cmap=True)

# JAXVacua
import jaxvacua as jvc

# CYTools
from cytools import Polytope, Cone, fetch_polytopes, read_polytopes

warnings.filterwarnings('ignore')

## CYTools introduction

[CYTools](https://cy.tools) is an open-source software package developed by Liam McAllister's group, designed for the analysis of Calabi-Yau manifolds derived from the [Kreuzer-Skarke database](http://hep.itp.tuwien.ac.at/~kreuzer/CY/CYcy.html). CYTools offers significantly enhanced computational performance compared to other software commonly used in this field.

The package is predominantly written in Python and interfaces seamlessly with various other open-source software. It is distributed as a Docker image encompassing all necessary dependencies, ensuring straightforward installation and usability across most operating systems.

The software provides efficient algorithms for handling and triangulating reflexive polytopes, facilitating the extraction of data associated with Calabi-Yau hypersurfaces. It is capable of exploring the entire Kreuzer-Skarke database, even for polytopes with the largest Hodge numbers.

## Grabbing models at LCS from the Kreuzer-Skarke list

### Getting started: loading geometries from CYTools

To begin with, let us grab CY geometries from the Kreuzer-Skarke list. We are using [CYTools](https://cy.tools) to fetch 4D reflexive polytopes via `cytools.fetch_polytopes` and to triangualte them. A CY hypersurface is then easily obtained as follows:

In [ ]:
p = fetch_polytopes(h11=2,h12=272,limit=5,lattice="N",as_list=True)[0]
cy = p.triangulate().get_cy()
cy

For more details, we refer to the documentation of [CYTools](https://cy.tools).

For our purposes, the object `cy` serves as the mirror dual CY whose topological data fixes the periods in the **Large Complex Structure** (LCS) limit
on the Type IIB side. Among others, we can compute the triple intersection numbers and second Chern class of the CY as follows:

In [ ]:
int_nums=cy.intersection_numbers(in_basis=True,format="dense")
sec_chern=cy.second_chern_class(in_basis=True)

Similarly, we obtain the Gopakumar-Vafa (GV) invariants through:

In [ ]:
gvs=cy.compute_gvs(max_deg=2).dok
gvs



Another useful quantitiy is the **Kähler cone** which can be easily obtained from CYTools:

In [ ]:
kahler_cone = cy.toric_kahler_cone()
kahler_cone

Its extremal rays, rays and bounding hyperplanes can be obtained as follows:

In [ ]:
extremal_rays=kahler_cone.extremal_rays()
rays=kahler_cone.rays()
hyperplanes = kahler_cone.extremal_hyperplanes()
extremal_rays,rays,hyperplanes

These objects will be useful when e.g. sampling points in moduli space.

A helpful reference point is the tip of the stretched Kähler cone:

In [ ]:
tip = kahler_cone.tip_of_stretched_cone(c=1)
tip

This point lies at distance $c=1$ from each of the walls as can be seen as follows:

In [ ]:
hyperplanes@tip

The tip typically serves as a reference point in the sense that computational control over quantum corrections can be ensured in its vicinity.

All of the above data is computed when we initialize a model with `limit="LCS"` and CYTools input, as we now illustrate.

### Using CYTools to construct new LCS models and model storage

Let us now illustrate how CYTools objects can be used in `JAXVacua`. 
By setting `use_cytools=True` and using the input `mirror_cy = cy`,
we can generate a `jaxvacua.flux_eft` object as follows

In [ ]:
model = jvc.FluxEFT(h12=cy.h11(), Q=cy.h11()+cy.h12()+2, model_type="KS", use_cytools=True, mirror_cy = cy)

In the background, we compute the required data for LCS limits such as

In [ ]:
model.lcs_tree.intnums, model.lcs_tree.a_matrix, model.lcs_tree.b_vector, model.lcs_tree.hyperplanes

By abuse of convention, it is strictly speaking not required to input a `cytools.CalabiYau` object. Instead, we can either just provide a `cytools.Polytope` object via `mirror_cy = p` or `cytools.Triangulation` object via `mirror_cy = p.triangulate()`. In the former case, we simply construct the CY hypersurface from the Delaunay triangulation of said polytope.

If a certain model is supposed to be used many times in the future, we can save the data in files in the designated model storage directory.
This is achieved by setting `save_file=True` and providing a `model_ID` which can be either a string or an integer:

In [ ]:
model = jvc.FluxEFT(h12=cy.h11(), 
                    Q=cy.h11()+cy.h12()+2, 
                    use_cytools=True, 
                    mirror_cy = cy, 
                    model_ID = "newton", 
                    save_file=True)

We can then load the model **without using CYTools** and without inputting `cy` by simply loading the corresponding file saved above:

In [ ]:
model_reload = jvc.FluxEFT(h12=cy.h11(), 
                           Q=cy.h11()+cy.h12()+2, 
                           use_cytools=False, 
                           mirror_cy = None, 
                           model_ID = "newton")

This is useful whenever models will be used across many different projects.

Note that CYTools also allows us to compute GV invariants as shown above. These can be included by setting the input parameter
`maximum_degree` to a non-zero value:

In [ ]:
model_inst = jvc.FluxEFT(h12=cy.h11(), 
                         Q=cy.h11()+cy.h12()+2, 
                         maximum_degree=10,
                         use_cytools=True, 
                         mirror_cy = cy)

model_inst


If we want to add the the GVs to our model storage, we can run similarly to above

In [ ]:
model_inst = jvc.FluxEFT(h12=cy.h11(), 
                         Q=cy.h11()+cy.h12()+2, 
                         maximum_degree=10,
                         use_cytools=True, 
                         mirror_cy = cy, 
                         model_ID = "newton", 
                         save_file=True)

model_inst

We are asked whether we are fine with overwriting the data for the provided `model_ID` because we already saved it above.

### Applying basis transformation

For many applications, it might be necessary to choose a specific basis of divisors for the CY. 
E.g. for models at $h^{1,2}=2$, we can always find a basis in which the Kähler cone is the entire positive quadrant.
Such a basis transformation is then obtained as follows:

In [ ]:
mcap = cy.mori_cone_cap(in_basis=True)
Kcup = mcap.dual_cone()
basis_trafo = Kcup.extremal_rays()
basis_trafo

By adding `basis_trafo` as input, we obtain a new `jaxvacua.flux_eft` objects where now all the relevant quantities have been
transformed to the desired basis:

In [ ]:
model_basis = jvc.FluxEFT(h12=cy.h11(), 
                          Q=cy.h11()+cy.h12()+2, 
                          model_type="KS", 
                          maximum_degree=10,
                          use_cytools=True, 
                          mirror_cy = cy, 
                          basis_change=basis_trafo)

model_basis

Explicitly, we have for the intersection numbers $\widetilde{\kappa}_{ijk}$, $a_{ij}$, $b_i$ and the Kähler cone hyperplanes

In [ ]:
model_basis.lcs_tree.intnums, model_basis.lcs_tree.a_matrix, model_basis.lcs_tree.b_vector, model_basis.lcs_tree.hyperplanes

This is precisely the basis used e.g. in [1912.10047](https://inspirehep.net/literature/1772253), [2306.06160](https://inspirehep.net/literature/2668029) or [2501.03984](https://inspirehep.net/literature/2865868).
Since `model_basis.periods.hyperplanes` corresponds to the canonical basis of $\mathbb{R}^2$, the Kähler cone indeed corresponds to the entire positive quadrant.

Similarly, the GV invariants and associated curve charges match those from [1912.10047](https://inspirehep.net/literature/1772253):

In [ ]:
model_basis.lcs_tree.gv_charges[:5],model_basis.lcs_tree.gv_invariants[:5],model_basis.lcs_tree.grading_vector

## Take-aways

- A `cytools.CalabiYau` object exposes the topological data — intersection numbers, GV invariants, Kähler cone, Mori cone — that JAXVacua needs to build an LCS prepotential.
- `jvc.FluxEFT(..., use_cytools=True, mirror_cy=cy)` ingests the CYTools object directly and stores the model on disk for re-use without CYTools.
- The Kähler cone tip and stretched-cone tip are useful reference points for ensuring computational control over instanton corrections.
- Passing `basis_trafo=…` rewrites intersection numbers, $a_{ij}$, $b_i$, hyperplanes, and GV charges in a chosen basis of divisors.
- Saved JAXVacua models can be reloaded across projects without re-running CYTools, which is the recommended workflow for downstream notebooks.

## Further reading

- [Notebook 02 — JAXVacua overview](02_jaxvacua_overview.ipynb)
- CICY tutorial — see the stringforge documentation
- [CYTools documentation](https://cy.tools)
- [arXiv:2211.03823](https://arxiv.org/abs/2211.03823) — CYTools reference paper